In [120]:
include("../src/EBC.jl")
include("../../BeamToyModel/src/BeamToyModel.jl")
using Base.Threads
using Statistics
using NPZ

In [19]:
nside_in = 32
res = Resolution(nside_in)
lmax_in = 3nside_in-1
fwhm = deg2rad(1.0)
blm_harmonic = BeamToyModel.gaussian_harmonic(lmax_in, fwhm, mmax = 2)
;

In [117]:
cs = ConvolutionSky(lmax = lmax_in, numofsky = 1)
@show fieldnames(typeof(cs))
cb = ConvolutionBeam(lmax = lmax_in, mmax =2, numofbeams = 1)
@show fieldnames(typeof(cb))
cc = ConvolutionCalculate(nside_output = nside_in, lstart = 0, mmax_calculate = 2)
@show fieldnames(typeof(cc))

fieldnames(typeof(cs)) = (:numofsky, :lmax, :alm)
fieldnames(typeof(cb)) = (:numofbeams, :lmax, :mmax, :blm)
fieldnames(typeof(cc)) = (:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)


(:nside_output, :lstart, :lstop, :mmax_calculate, :HWP)

In [103]:
cc.lstop

95

In [21]:
#cb.mmax = 2
cb.blm[:,:,1] = hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)
;

In [123]:
alm_in = npzread("./alm=CMB=r0=nside32=seed1.npy")
cs.alm[:,:,1] = alm_in

3×4656 Matrix{ComplexF64}:
 0.0+0.0im  0.0+0.0im     -17.2712+0.0im  …     0.24645+1.48487im
 0.0+0.0im  0.0+0.0im    -0.216198+0.0im      0.0058999-0.00792962im
 0.0+0.0im  0.0+0.0im  -0.00188638+0.0im     4.92514e-5-0.00116108im

In [26]:
pix_idx = 6
θ_pix,φ_pix = Healpix.pix2angRing(res, pix_idx)

(0.05103657515266638, 1.1780972450961724)

In [55]:
nptg = 10
θ = θ_pix
φ = φ_pix
dθ = rand(nptg)*1e-2*0
dφ = rand(nptg)*1e-2*1
ψ = rand(nptg)*2pi*0

10-element Vector{Float64}:
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [75]:
function calc_local_euiler_angles(res, pix_idx, φ, θ, ψ)
    alphas = zeros(size(θ))
    betas = zeros(size(θ))
    gammas = zeros(size(θ))
    θ_pix,φ_pix = Healpix.pix2angRing(res, pix_idx)
    dθ = θ .- θ_pix
    dφ = φ .- φ_pix
    for i in eachindex(θ)
        err, (alphas[i], betas[i], gammas[i]) = check_split(φ_pix, θ_pix, dφ[i], dθ[i], ψ[i])
    end
    return alphas, betas, gammas
end

calc_local_euiler_angles (generic function with 1 method)

In [76]:
α_local, β_local, γ_local = calc_local_euiler_angles(res, pix_idx, φ.+dφ, θ.+dθ, ψ)
;

In [79]:
@time A = local_effective_wignerD(cb, cc, α_local, β_local, γ_local)
@time A_conj = local_effective_wignerD_conj(cb, cc, α_local, β_local, γ_local)
@time A2_conj = local_effective_wignerD_conj_reduced(cb, cc, α_local, β_local, γ_local, τ=5)
;

  0.775745 seconds (27 allocations: 3.442 MiB)
  0.683399 seconds (27 allocations: 3.442 MiB)
  0.070069 seconds (23 allocations: 493.266 KiB)


In [129]:
A_conj

9216×474 SparseMatrixCSC{ComplexF64, Int64} with 46070 stored entries:
⎡⢧⠀⠀⎤
⎢⢸⠀⠀⎥
⎢⢸⠀⠀⎥
⎢⢸⡄⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⡇⠀⎥
⎢⠀⢿⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⠀⎥
⎢⠀⢸⡀⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎢⠀⠀⡇⎥
⎣⠀⠀⡇⎦

In [80]:
B = global_wignerD_conj(cc, φ, θ, 0)

9216×9216 SparseMatrixCSC{ComplexF64, Int64} with 1179615 stored entries:
⎡⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠈⠻⢆⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠛⣤⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠿⣧⣀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⣀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠻⣦⡄⠀⎥
⎣⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢻⣶⎦

In [125]:
blm_ = slice_spin_blm_by_l(cb, cc)
alm_ = slice_spin_alm_by_l(cs, cc)

9216×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
        0.0+0.0im              -0.0-0.0im                -0.0-0.0im
       -0.0+0.0im               0.0-0.0im                 0.0-0.0im
        0.0+0.0im              -0.0-0.0im                -0.0-0.0im
        0.0+0.0im              -0.0-0.0im                -0.0-0.0im
    8.54357+5.79895im     -0.111944+0.0620931im     -0.114329+0.0641405im
   -1.00807-3.35171im    -0.0881878+0.186775im     -0.0869968+0.185436im
   -17.2712+0.0im          0.216198+0.00188638im     0.216198-0.00188638im
    1.00807-3.35171im     0.0869968+0.185436im      0.0881878+0.186775im
    8.54357-5.79895im     -0.114329-0.0641405im     -0.111944-0.0620931im
   -3.35884+3.03626im     0.0491135-0.0154069im     0.0501488-0.0154036im
    30.3017+6.43539im    0.00841299+0.0250256im    0.00721439+0.0232682im
    9.86375-8.44165im     -0.103262-0.0319001im     -0.104217-0.032071im
   -24.1408+0.0im         -0.136862+0.00101255im    -0.136862-0.00101255im
           ⋮        

In [127]:
alm_[:,1,1]

9216-element Vector{ComplexF64}:
                  0.0 + 0.0im
                 -0.0 + 0.0im
                  0.0 + 0.0im
                  0.0 + 0.0im
    8.543568995773878 + 5.798947598191975im
  -1.0080652144263396 - 3.3517144478685674im
  -17.271243915162433 + 0.0im
   1.0080652144263396 - 3.3517144478685674im
    8.543568995773878 - 5.798947598191975im
  -3.3588365694499154 + 3.0362642405627396im
   30.301692104571522 + 6.4353923531426815im
    9.863748605257921 - 8.441650814711696im
  -24.140804211507337 + 0.0im
                      ⋮
    1.323217185999896 + 1.8139986289155727im
   0.9308999961531088 + 0.48772938488009254im
 -0.09091866343533628 + 0.5150974224556036im
  -0.2711615606820372 - 1.4197171937851116im
  0.23187692582352853 - 0.5770128372832399im
  -1.3900746391622874 + 1.1640581417802767im
    0.684979526980874 - 0.7479963922160988im
   1.1685455425448232 + 1.0373772532773402im
   0.4347824552429817 - 0.4311730115832015im
  -0.3625393546769521 - 1.08209723683912im
  

In [128]:
blm_

474×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
     ⋮                   
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0+1.0im  -2.0-0.0im  -0.0-2.0im
 -1.0-1.0im   2.0+0.0im   0.0+2.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im
  1.0-1.0im  -0.0+2.0im  -2.0+0.0im

In [105]:
idx_in = alm_idx(l=cb.lmax, m=cb.lmax, lmax=cb.lmax, mmax = cb.mmax)

LoadError: ArgumentError: m must be in [0, mmax]

In [104]:
cb.mmax

2

In [ ]:
n = alm_idx(lmax = cb.lmax, mmax = cc.mmax_calculate, l=cb.lmax, m=cc.mmax_calculate)
blm_calc = Array{ComplexF64,3}(undef, n, 3, cb.numofbeams)

285×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
  2.2749e-314+6.92672e-310im  …  6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92669e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92669e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im  …  6.92672e-310+6.92672e-310im
          NaN+4.0e-323im         6.92672e-310+6.92672e-310im
 2.27484e-314+6.92672e-310im     6.92672e-310+6.92672e-310im
 2.27488e-314+6.92672e-310im     6.92672e-310+6.92672e-310im
          NaN+6.92672e-310im     6.92672e-310+6.92664e-310im
 2.27484e-314+6.92672e-310im  …  6.92672e-310+6.92672e-310im
 2.27484e-314+0.0im              6.92664e-310+6.92672e-310im
 2.27484e-314+6.92672e-310im     6.92672e-310+6.92672e-310im
             ⋮                ⋱  
     4.0e-323+0.0im              6.92672e-310+1.04e-322im
          NaN+1.50662e-312im     6.92672e-310+1.7e-322im
          0.0+6.

In [88]:
n = 10
blm_calc = Array{ComplexF64,3}(undef, 3, n, cb.numofbeams)


3×10×1 Array{ComplexF64, 3}:
[:, :, 1] =
 2.44636e-314+2.43256e-314im  …  2.45103e-314+2.45103e-314im
 2.43299e-314+2.46808e-314im     2.77884e-314+2.46808e-314im
 2.46808e-314+2.46808e-314im              0.0+2.78145e-314im

In [99]:
cb.blm

285×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
 0.282095+0.0im       0.0+0.0im  0.0+0.0im
 0.488576+0.0im       0.0+0.0im  0.0+0.0im
 0.630679+0.0im       0.0+0.0im  0.0+0.0im
 0.746107+0.0im       0.0+0.0im  0.0+0.0im
  0.84582+0.0im       0.0+0.0im  0.0+0.0im
 0.934832+0.0im       0.0+0.0im  0.0+0.0im
  1.01593+0.0im       0.0+0.0im  0.0+0.0im
  1.09087+0.0im       0.0+0.0im  0.0+0.0im
  1.16081+0.0im       0.0+0.0im  0.0+0.0im
  1.22659+0.0im       0.0+0.0im  0.0+0.0im
  1.28882+0.0im       0.0+0.0im  0.0+0.0im
  1.34798+0.0im       0.0+0.0im  0.0+0.0im
  1.40444+0.0im       0.0+0.0im  0.0+0.0im
         ⋮                       
      0.0+0.0im  -1.50692+0.0im  0.0-1.50692im
      0.0+0.0im  -1.50875+0.0im  0.0-1.50875im
      0.0+0.0im  -1.51039+0.0im  0.0-1.51039im
      0.0+0.0im  -1.51186+0.0im  0.0-1.51186im
      0.0+0.0im  -1.51314+0.0im  0.0-1.51314im
      0.0+0.0im  -1.51424+0.0im  0.0-1.51424im
      0.0+0.0im  -1.51517+0.0im  0.0-1.51517im
      0.0+0.0im  -1.51592+0.0im  0.0

In [91]:
cb.blm

285×3×1 Array{ComplexF64, 3}:
[:, :, 1] =
 0.282095+0.0im       0.0+0.0im  0.0+0.0im
 0.488576+0.0im       0.0+0.0im  0.0+0.0im
 0.630679+0.0im       0.0+0.0im  0.0+0.0im
 0.746107+0.0im       0.0+0.0im  0.0+0.0im
  0.84582+0.0im       0.0+0.0im  0.0+0.0im
 0.934832+0.0im       0.0+0.0im  0.0+0.0im
  1.01593+0.0im       0.0+0.0im  0.0+0.0im
  1.09087+0.0im       0.0+0.0im  0.0+0.0im
  1.16081+0.0im       0.0+0.0im  0.0+0.0im
  1.22659+0.0im       0.0+0.0im  0.0+0.0im
  1.28882+0.0im       0.0+0.0im  0.0+0.0im
  1.34798+0.0im       0.0+0.0im  0.0+0.0im
  1.40444+0.0im       0.0+0.0im  0.0+0.0im
         ⋮                       
      0.0+0.0im  -1.50692+0.0im  0.0-1.50692im
      0.0+0.0im  -1.50875+0.0im  0.0-1.50875im
      0.0+0.0im  -1.51039+0.0im  0.0-1.51039im
      0.0+0.0im  -1.51186+0.0im  0.0-1.51186im
      0.0+0.0im  -1.51314+0.0im  0.0-1.51314im
      0.0+0.0im  -1.51424+0.0im  0.0-1.51424im
      0.0+0.0im  -1.51517+0.0im  0.0-1.51517im
      0.0+0.0im  -1.51592+0.0im  0.0

In [61]:
φ.+dφ

10-element Vector{Float64}:
 1.184885942919513
 1.1866962103042489
 1.1784405008562786
 1.1782985326275113
 1.1790953864088611
 1.1839966137867675
 1.1824868418720687
 1.187186938843399
 1.1793807964482965
 1.1837689100106155

In [71]:
alphas = zeros(size(dθ))
betas = zeros(size(dθ))
gammas = zeros(size(dθ))

for i in eachindex(dθ)
    err, (alphas[i], betas[i], gammas[i]) = check_split(φ, θ, dφ[i], dθ[i], ψ[i])
    if err >= 1e0
        @show err
    end
end

In [73]:
@show α_local == alphas
@show β_local == betas
@show γ_local == gammas

α_local == alphas = true
β_local == betas = true
γ_local == gammas = true


true

In [50]:
@time A = local_effective_wignerD(cb, cc, alphas, betas, gammas)
@time A_conj = local_effective_wignerD_conj(cb, cc, alphas, betas, gammas)
@time A2_conj = local_effective_wignerD_conj_reduced(cb, cc, alphas, betas, gammas; τ=5)
;

  0.695295 seconds (27 allocations: 3.442 MiB)
  0.788696 seconds (27 allocations: 3.442 MiB)
  0.081685 seconds (23 allocations: 493.266 KiB)


In [51]:
@show maximum(abs.(A_conj .- conj.(A)))
@show maximum(abs.(A2_conj .- conj.(A)))
@show maximum(abs.(A_conj .- (A2_conj) ))

maximum(abs.(A_conj .- conj.(A))) = 2.374296708786948e-14
maximum(abs.(A2_conj .- conj.(A))) = 3.935821241779829e-14
maximum(abs.(A_conj .- A2_conj)) = 3.935821241779829e-14


3.935821241779829e-14

In [10]:
hcat(blm_harmonic.blmT.alm,blm_harmonic.blmE.alm,blm_harmonic.blmB.alm)

285×3 Matrix{ComplexF64}:
 0.282095+0.0im       0.0+0.0im  0.0+0.0im
 0.488576+0.0im       0.0+0.0im  0.0+0.0im
 0.630679+0.0im       0.0+0.0im  0.0+0.0im
 0.746107+0.0im       0.0+0.0im  0.0+0.0im
  0.84582+0.0im       0.0+0.0im  0.0+0.0im
 0.934832+0.0im       0.0+0.0im  0.0+0.0im
  1.01593+0.0im       0.0+0.0im  0.0+0.0im
  1.09087+0.0im       0.0+0.0im  0.0+0.0im
  1.16081+0.0im       0.0+0.0im  0.0+0.0im
  1.22659+0.0im       0.0+0.0im  0.0+0.0im
  1.28882+0.0im       0.0+0.0im  0.0+0.0im
  1.34798+0.0im       0.0+0.0im  0.0+0.0im
  1.40444+0.0im       0.0+0.0im  0.0+0.0im
         ⋮                       
      0.0+0.0im  -1.50692+0.0im  0.0-1.50692im
      0.0+0.0im  -1.50875+0.0im  0.0-1.50875im
      0.0+0.0im  -1.51039+0.0im  0.0-1.51039im
      0.0+0.0im  -1.51186+0.0im  0.0-1.51186im
      0.0+0.0im  -1.51314+0.0im  0.0-1.51314im
      0.0+0.0im  -1.51424+0.0im  0.0-1.51424im
      0.0+0.0im  -1.51517+0.0im  0.0-1.51517im
      0.0+0.0im  -1.51592+0.0im  0.0-1.51592im
     

3×1149×1 Array{ComplexF64, 3}:
[:, :, 1] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im  …  1.0+1.0im  1.0+1.0im  1.0+1.0im
 1.0+1.0im  1.0+1.0im  1.0+1.0im     1.0+1.0im  1.0+1.0im  1.0+1.0im
 1.0+1.0im  1.0+1.0im  1.0+1.0im     1.0+1.0im  1.0+1.0im  1.0+1.0im

# Convolution test

In [ ]:
@inline function convolver_(alm, blm, wignerD)
    
    for l in 0:cp.lmax
        result_1 += transpose(alm_new[1,l+1,lmax+1-l:lmax+1+l])*conj(eff_wignerD[l+1][:,:]*blm_new[1,l+1,lmax+1-l:lmax+1+l])
        result_2 += transpose(alm_new[2,l+1,lmax+1-l:lmax+1+l])*conj(eff_wignerD[l+1][:,:]*blm_new[2,l+1,lmax+1-l:lmax+1+l])
        result_3 += transpose(alm_new[3,l+1,lmax+1-l:lmax+1+l])*conj(eff_wignerD[l+1][:,:]*blm_new[3,l+1,lmax+1-l:lmax+1+l])
    end

end



convolver_ (generic function with 1 method)

In [15]:
cb.blm

1×3×1149 Array{ComplexF64, 3}:
[:, :, 1] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 2] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 3] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

;;; … 

[:, :, 1147] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1148] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1149] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

In [8]:
cb.lmax

383

In [13]:
alm_idx(lmax = cb.lmax,mmax =2, l=cb.lmax, m=2)

1149

In [5]:
cb.blm

1×3×1149 Array{ComplexF64, 3}:
[:, :, 1] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 2] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 3] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

;;; … 

[:, :, 1147] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1148] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

[:, :, 1149] =
 1.0+1.0im  1.0+1.0im  1.0+1.0im

In [12]:
cs.alm

2×2 Matrix{ComplexF64}:
 1.0+1.0im  1.0+1.0im
 1.0+1.0im  1.0+1.0im